# Power BI Dashboard Preparation

## Objective

After completing data cleaning and exploratory data analysis, the final dataset needs to be prepared specifically for dashboard development.

This notebook validates all business metrics, verifies KPI calculations, prepares dashboard dimensions, and exports the final dataset used inside Power BI.

Unlike previous notebooks, no additional cleaning is performed here.

The focus is entirely on dashboard readiness and metric validation.

In [2]:
# ==========================================================
# Import Libraries
# ==========================================================

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

In [3]:
# ==========================================================
# Load Clean Dataset
# ==========================================================

df = pd.read_csv("../data/cleaned_flipkart_dataset.csv")

print(df.shape)

df.head()

(15827, 10)


,brand,product_name,product_category_tree,product_rating,retail_price,discounted_price,discount_percentage,main_category,sub_category,price_bucket
0,Alisha,Alisha Solid Women's Cycling Shorts,"[""Clothing >> Women's Clothing >> Lingerie, Sl...",NaN,999.0,379.0,62.062062,Clothing,Women's Clothing,<₹500
1,FabHomeDecor,FabHomeDecor Fabric Double Sofa Bed,"[""Furniture >> Living Room Furniture >> Sofa B...",NaN,32157.0,22646.0,29.576764,Furniture,Living Room Furniture,₹5K+
2,AW,AW Bellies,"[""Footwear >> Women's Footwear >> Ballerinas >...",NaN,999.0,499.0,50.050050,Footwear,Women's Footwear,<₹500
3,Alisha,Alisha Solid Women's Cycling Shorts,"[""Clothing >> Women's Clothing >> Lingerie, Sl...",NaN,699.0,267.0,61.802575,Clothing,Women's Clothing,<₹500
4,Sicons,Sicons All Purpose Arnica Dog Shampoo,"[""Pet Supplies >> Grooming >> Skin & Coat Care...",NaN,220.0,210.0,4.545455,Pet Supplies,Grooming,<₹500


## Dataset Validation

Before building the dashboard, the exported dataset is validated to ensure that:

- all engineered columns are present
- no unexpected missing values exist
- KPI calculations remain consistent with the analysis notebook

This prevents inconsistencies between Python analysis and Power BI visuals.

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15827 entries, 0 to 15826
Data columns (total 10 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   brand                  15827 non-null  object 
 1   product_name           15827 non-null  object 
 2   product_category_tree  15827 non-null  object 
 3   product_rating         1620 non-null   float64
 4   retail_price           15827 non-null  float64
 5   discounted_price       15827 non-null  float64
 6   discount_percentage    15827 non-null  float64
 7   main_category          15827 non-null  object 
 8   sub_category           15571 non-null  object 
 9   price_bucket           15827 non-null  object 
dtypes: float64(4), object(6)
memory usage: 1.2+ MB


In [5]:
df.isnull().sum()

brand                        0
product_name                 0
product_category_tree        0
product_rating           14207
retail_price                 0
discounted_price             0
discount_percentage          0
main_category                0
sub_category               256
price_bucket                 0
dtype: int64

## Validation Result

The cleaned dataset contains only expected missing values.

The only remaining missing values belong to **Product Rating**, which is intentionally retained because the original dataset contained very limited rating information.

All pricing variables, category variables and engineered columns are complete and ready for visualization.

# Dashboard KPI Validation

Before designing dashboard cards, every KPI is validated inside Python.

This ensures that Power BI produces the same business values.

In [6]:
# ==========================================================
# KPI Validation
# ==========================================================

print("Total Products :", len(df))

print("")

print("Average Retail Price :")

print(round(df["retail_price"].mean(),2))

print("")

print("Average Discount % :")

print(round(df["discount_percentage"].mean(),2))

print("")

print("Total Categories :")

print(df["main_category"].nunique())

print("")

print("Total Brands :")

print(
    df[
        ~df["brand"].isin(["Unknown","Generic"])
    ]["brand"].nunique()
)

Total Products : 15827

Average Retail Price :
3324.56

Average Discount % :
39.41

Total Categories :
251

Total Brands :
3485


In [7]:
# ==========================================================
# Dashboard Data Quality KPIs
# ==========================================================

unknown_brand_percentage = (
    (df["brand"]=="Unknown").mean()*100
)

rating_percentage = (
    df["product_rating"].notna().mean()*100
)

print(f"Unknown Brand % : {unknown_brand_percentage:.2f}%")

print(f"Products with Rating : {rating_percentage:.2f}%")

Unknown Brand % : 28.90%
Products with Rating : 10.24%


## KPI Validation Summary

The validated metrics above become the KPI cards displayed in the Power BI dashboard.

Using Python for verification ensures that dashboard calculations remain accurate and reproducible.

In [8]:
# ==========================================================
# Dashboard Filters
# ==========================================================

print("Main Categories")

print(df["main_category"].nunique())

print()

print("Price Buckets")

print(df["price_bucket"].value_counts())

print()

print("Brands")

print(
    df[
        ~df["brand"].isin(["Unknown","Generic"])
    ]["brand"].nunique()
)

Main Categories
251

Price Buckets
price_bucket
<₹500      7471
₹500–1K    4161
₹1K–2K     2411
₹5K+       1016
₹2K–5K      768
Name: count, dtype: int64

Brands
3485


In [9]:
price_order = {
    "<₹500": 1,
    "₹500–1K": 2,
    "₹1K–2K": 3,
    "₹2K–5K": 4,
    "₹5K+": 5
}

df["price_bucket_sort"] = df["price_bucket"].map(price_order)

In [10]:
df["discount_bucket"] = pd.cut(
    df["discount_percentage"],
    bins=[0,10,20,40,60,80,100],
    labels=[
        "0–10%",
        "10–20%",
        "20–40%",
        "40–60%",
        "60–80%",
        "80%+"
    ],
    include_lowest=True
)

In [11]:
discount_order = {
    "0–10%":1,
    "10–20%":2,
    "20–40%":3,
    "40–60%":4,
    "60–80%":5,
    "80%+":6
}

df["discount_bucket_sort"] = df["discount_bucket"].map(discount_order)

# Dashboard Relationships

The dashboard is designed around three primary dimensions:

- Product Category
- Price Segment
- Brand

Each visual interacts dynamically through slicers, enabling users to explore pricing strategies across different segments of the catalogue.

In [12]:
# ==========================================================
# Export Final Dashboard Dataset
# ==========================================================

df.to_csv(
    "../data/dashboard_dataset.csv",
    index=False
)

print("Dashboard dataset exported successfully.")

Dashboard dataset exported successfully.


# Dashboard Ready

The exported dataset contains:

- Cleaned records
- Engineered variables
- Dashboard dimensions
- Pricing metrics
- Discount metrics
- Product segmentation variables

This dataset serves as the single source of truth for the Power BI dashboard.